# 02 Experiments


## Section 2.1 - Setup


In [ ]:
import logging

import lakefs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from IPython.display import display

from helpers import (
    compute_binary_classification_metrics,
    create_branch,
    create_xgboost_classifier,
    engineer_time_features,
    engineer_route_features,
    frequency_encode,
    compute_delay_rates,
    apply_delay_rates,
    lakefs_commit,
    load_metrics_json,
    load_predictions_parquet,
    plot_confusion_matrix,
    plot_feature_importance,
    plot_precision_recall_curve,
    read_parquet,
    save_metrics_json,
    save_predictions_parquet,
    temporal_train_test_split,
    write_parquet,
)
from notebook_setup import build_notebook_config, initialize_lakefs_repository

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
LOGGER = logging.getLogger("02_experiments")


In [ ]:
config = build_notebook_config()
init_result = initialize_lakefs_repository(config)
print(init_result.message)

lakefs_client = lakefs.Client(
    host=config.endpoint,
    username=config.access_key,
    password=config.secret_key,
)

LOGGER.info("Loading silver dataset from branch-aware path")
silver_repo = lakefs.Repository(config.repo_name, client=lakefs_client)
silver_obj = silver_repo.branch("silver").object("silver/flights_2023_clean.parquet")
with silver_obj.reader(mode="rb") as reader:
    silver_df = pd.read_parquet(reader)

print("silver shape:", silver_df.shape)
display(silver_df.head())


## Section 2.2 - Experiment A: Time-Based Features

Engineer time-based features (hour, day-of-week, month, weekend, holiday, time bucket),
label-encode categoricals, train XGBoost, and persist gold artifacts to lakeFS.


In [ ]:
# --- 2.2a: Create experiment branch and engineer time features ---
BRANCH_TIME = "experiment-time-features"
create_branch(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_TIME,
    source_branch="silver",
)
print(f"Branch '{BRANCH_TIME}' ready (from silver)")

# Engineer time features from silver dataset
time_df = engineer_time_features(silver_df)
print("Time features engineered, shape:", time_df.shape)
display(time_df[["FL_DATE", "DEP_TIME", "hour_of_day", "day_of_week", "month",
                  "is_weekend", "is_holiday_period", "time_of_day_bucket"]].head(10))


In [ ]:
# --- 2.2b: Label-encode categoricals and build feature matrix ---

# Label-encode categorical columns for Experiment A
le_airline = LabelEncoder()
le_origin = LabelEncoder()
le_bucket = LabelEncoder()

time_df["airline_enc"] = le_airline.fit_transform(time_df["AIRLINE_CODE"].astype(str))
time_df["origin_enc"] = le_origin.fit_transform(time_df["ORIGIN"].astype(str))
time_df["bucket_enc"] = le_bucket.fit_transform(time_df["time_of_day_bucket"].astype(str))

# Define feature columns for Experiment A
TIME_FEATURE_COLS = [
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "is_holiday_period",
    "bucket_enc",
    "airline_enc",
    "origin_enc",
    "DISTANCE",
]
TARGET_COL = "is_delayed"

features_time = time_df[TIME_FEATURE_COLS + [TARGET_COL, "FL_DATE"]].copy()
print("Feature matrix shape:", features_time.shape)
print("Feature columns:", TIME_FEATURE_COLS)
print("Class balance:")
print(features_time[TARGET_COL].value_counts(normalize=True))


In [ ]:
# --- 2.2c: Temporal split and persist gold feature artifact ---
CUTOFF_DATE = "2023-11-01"
X_train_t, X_test_t, y_train_t, y_test_t = temporal_train_test_split(
    features_time, target_col=TARGET_COL, date_col="FL_DATE", cutoff_date=CUTOFF_DATE
)
print(f"Temporal split (cutoff={CUTOFF_DATE}): Train: {X_train_t.shape}, Test: {X_test_t.shape}")
print(f"Train delay rate: {y_train_t.mean():.4f}, Test delay rate: {y_test_t.mean():.4f}")

# Write gold feature artifact to lakeFS
write_parquet(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_TIME,
    path="gold/features_time.parquet",
    df=features_time,
)
lakefs_commit(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_TIME,
    message="Gold layer: time-based features",
    metadata={"phase": "phase-5", "layer": "gold", "experiment": "time"},
)
print("gold/features_time.parquet committed on", BRANCH_TIME)


In [ ]:
# --- 2.2d: Train XGBoost model ---
model_time = create_xgboost_classifier(random_state=42)
model_time.fit(X_train_t, y_train_t)
print("XGBoost (time) training complete")

# Predict
y_pred_t = model_time.predict(X_test_t)
y_scores_t = model_time.predict_proba(X_test_t)[:, 1]

# Compute metrics
metrics_time = compute_binary_classification_metrics(y_test_t, y_pred_t, y_scores_t)
print("\n=== Experiment A (Time Features) Metrics ===")
for k, v in metrics_time.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
# --- 2.2e: Generate and save evaluation visuals ---
cm_path = plot_confusion_matrix(y_test_t, y_pred_t, "confusion_matrix_time.png")
print("Saved:", cm_path)

pr_path = plot_precision_recall_curve(y_test_t, y_scores_t, "pr_curve_time.png")
print("Saved:", pr_path)

fi_names = list(X_train_t.columns)
fi_values = list(model_time.feature_importances_)
fi_path = plot_feature_importance(fi_names, fi_values, "feature_importance_time.png", top_n=15)
print("Saved:", fi_path)


In [ ]:
# --- 2.2f: Persist metrics + predictions to lakeFS and commit ---
import tempfile, os

# Save locally then upload
with tempfile.TemporaryDirectory() as tmpdir:
    # Metrics JSON
    metrics_local = os.path.join(tmpdir, "metrics_time.json")
    save_metrics_json(metrics_time, metrics_local)
    with open(metrics_local, "rb") as f:
        repo = lakefs.Repository(config.repo_name, client=lakefs_client)
        repo.branch(BRANCH_TIME).object("gold/metrics_time.json").upload(
            data=f.read(), mode="wb", content_type="application/json"
        )

    # Predictions parquet
    preds_df = pd.DataFrame({"y_true": y_test_t.values, "y_scores": y_scores_t})
    preds_local = os.path.join(tmpdir, "predictions_time.parquet")
    save_predictions_parquet(preds_df, preds_local)
    write_parquet(
        client=lakefs_client,
        repo_name=config.repo_name,
        branch_name=BRANCH_TIME,
        path="gold/predictions_time.parquet",
        df=preds_df,
    )

ref_time = lakefs_commit(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_TIME,
    message="Train XGBoost on time-based features, save metrics",
    metadata={"phase": "phase-5", "layer": "gold", "experiment": "time"},
)
print("Experiment A committed:", getattr(ref_time, "id", ref_time))
print("Metrics:", metrics_time)


## Section 2.3 - Experiment B: Route-Based Features

Engineer route/geography features with frequency encoding for high-cardinality
categoricals and leakage-safe delay-rate features computed from the training split only.


In [ ]:
# --- 2.3a: Create experiment branch and engineer route features ---
BRANCH_ROUTE = "experiment-route-features"
create_branch(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_ROUTE,
    source_branch="silver",
)
print(f"Branch '{BRANCH_ROUTE}' ready (from silver)")

# Engineer route features from silver dataset
route_df = engineer_route_features(silver_df)
print("Route features engineered, shape:", route_df.shape)
display(route_df[["ORIGIN", "DEST", "route", "DISTANCE", "distance_bucket"]].head(10))


In [ ]:
# --- 2.3b: Temporal split FIRST, then compute leakage-safe features from train only ---
TARGET_COL = "is_delayed"
CUTOFF_DATE = "2023-11-01"

# We need to split before computing delay rates and frequency encodings
# to prevent data leakage from test into training statistics.
# Include FL_DATE for the temporal split, then drop it afterward.
route_split_df = route_df[["FL_DATE", "AIRLINE_CODE", "ORIGIN", "DEST", "route",
                           "DISTANCE", "distance_bucket", TARGET_COL]].copy()

# Convert categoricals to string for encoding steps
for col in ["AIRLINE_CODE", "ORIGIN", "DEST"]:
    route_split_df[col] = route_split_df[col].astype(str)

# Temporal split — train on data before cutoff, test on data at/after cutoff
# temporal_train_test_split drops FL_DATE and TARGET_COL from the returned X frames
X_train_raw, X_test_raw, y_train_r, y_test_r = temporal_train_test_split(
    route_split_df, target_col=TARGET_COL, date_col="FL_DATE", cutoff_date=CUTOFF_DATE
)
print(f"Temporal split (cutoff={CUTOFF_DATE}) — Train: {X_train_raw.shape}, Test: {X_test_raw.shape}")
print(f"Train delay rate: {y_train_r.mean():.4f}, Test delay rate: {y_test_r.mean():.4f}")


In [ ]:
# --- 2.3c: Compute leakage-safe delay rates from train split only ---
train_with_target = X_train_raw.copy()
train_with_target[TARGET_COL] = y_train_r

origin_rates = compute_delay_rates(train_with_target, group_col="ORIGIN", target_col=TARGET_COL)
airline_rates = compute_delay_rates(train_with_target, group_col="AIRLINE_CODE", target_col=TARGET_COL)
route_rates = compute_delay_rates(train_with_target, group_col="route", target_col=TARGET_COL)

# Global train delay rate as fallback for unseen categories
global_delay_rate = float(y_train_r.mean())
print(f"Global train delay rate (fallback): {global_delay_rate:.4f}")
print(f"Unique origins in rates: {len(origin_rates)}, airlines: {len(airline_rates)}, routes: {len(route_rates)}")

# Apply rates to train and test
X_train_raw["origin_delay_rate"] = apply_delay_rates(X_train_raw, origin_rates, col="ORIGIN", default_rate=global_delay_rate)
X_train_raw["airline_delay_rate"] = apply_delay_rates(X_train_raw, airline_rates, col="AIRLINE_CODE", default_rate=global_delay_rate)
X_train_raw["route_delay_rate"] = apply_delay_rates(X_train_raw, route_rates, col="route", default_rate=global_delay_rate)

X_test_raw["origin_delay_rate"] = apply_delay_rates(X_test_raw, origin_rates, col="ORIGIN", default_rate=global_delay_rate)
X_test_raw["airline_delay_rate"] = apply_delay_rates(X_test_raw, airline_rates, col="AIRLINE_CODE", default_rate=global_delay_rate)
X_test_raw["route_delay_rate"] = apply_delay_rates(X_test_raw, route_rates, col="route", default_rate=global_delay_rate)

print("Delay-rate features applied to train and test.")


In [ ]:
# --- 2.3d: Frequency-encode high-cardinality categoricals ---
FREQ_ENCODE_COLS = ["AIRLINE_CODE", "ORIGIN", "DEST", "route"]

for col in FREQ_ENCODE_COLS:
    train_enc, test_enc = frequency_encode(X_train_raw[col], X_test_raw[col])
    X_train_raw[f"{col}_freq"] = train_enc
    X_test_raw[f"{col}_freq"] = test_enc

# Encode distance_bucket as ordinal
bucket_map = {"short": 0, "medium": 1, "long": 2, "very_long": 3}
X_train_raw["distance_bucket_enc"] = X_train_raw["distance_bucket"].map(bucket_map).fillna(0).astype(int)
X_test_raw["distance_bucket_enc"] = X_test_raw["distance_bucket"].map(bucket_map).fillna(0).astype(int)

# Final feature columns for Experiment B
ROUTE_FEATURE_COLS = [
    "DISTANCE",
    "distance_bucket_enc",
    "AIRLINE_CODE_freq",
    "ORIGIN_freq",
    "DEST_freq",
    "route_freq",
    "origin_delay_rate",
    "airline_delay_rate",
    "route_delay_rate",
]

X_train_r = X_train_raw[ROUTE_FEATURE_COLS].copy()
X_test_r = X_test_raw[ROUTE_FEATURE_COLS].copy()

print("Feature columns:", ROUTE_FEATURE_COLS)
print(f"Final train: {X_train_r.shape}, test: {X_test_r.shape}")
display(X_train_r.head())


In [ ]:
# --- 2.3e: Persist gold feature artifact and commit ---
features_route = pd.concat([X_train_r, X_test_r], axis=0)
features_route[TARGET_COL] = pd.concat([y_train_r, y_test_r], axis=0)

write_parquet(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_ROUTE,
    path="gold/features_route.parquet",
    df=features_route,
)
lakefs_commit(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_ROUTE,
    message="Gold layer: route-based features",
    metadata={"phase": "phase-6", "layer": "gold", "experiment": "route"},
)
print("gold/features_route.parquet committed on", BRANCH_ROUTE)


In [ ]:
# --- 2.3f: Train XGBoost model (same hyperparameters as Experiment A) ---
model_route = create_xgboost_classifier(random_state=42)
model_route.fit(X_train_r, y_train_r)
print("XGBoost (route) training complete")

# Predict
y_pred_r = model_route.predict(X_test_r)
y_scores_r = model_route.predict_proba(X_test_r)[:, 1]

# Compute metrics
metrics_route = compute_binary_classification_metrics(y_test_r, y_pred_r, y_scores_r)
print("\n=== Experiment B (Route Features) Metrics ===")
for k, v in metrics_route.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
# --- 2.3g: Generate and save evaluation visuals ---
cm_path_r = plot_confusion_matrix(y_test_r, y_pred_r, "confusion_matrix_route.png")
print("Saved:", cm_path_r)

pr_path_r = plot_precision_recall_curve(y_test_r, y_scores_r, "pr_curve_route.png")
print("Saved:", pr_path_r)

fi_names_r = list(X_train_r.columns)
fi_values_r = list(model_route.feature_importances_)
fi_path_r = plot_feature_importance(fi_names_r, fi_values_r, "feature_importance_route.png", top_n=15)
print("Saved:", fi_path_r)


In [ ]:
# --- 2.3h: Persist metrics + predictions to lakeFS and commit ---
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    # Metrics JSON
    metrics_local_r = os.path.join(tmpdir, "metrics_route.json")
    save_metrics_json(metrics_route, metrics_local_r)
    with open(metrics_local_r, "rb") as f:
        repo = lakefs.Repository(config.repo_name, client=lakefs_client)
        repo.branch(BRANCH_ROUTE).object("gold/metrics_route.json").upload(
            data=f.read(), mode="wb", content_type="application/json"
        )

    # Predictions parquet
    preds_df_r = pd.DataFrame({"y_true": y_test_r.values, "y_scores": y_scores_r})
    preds_local_r = os.path.join(tmpdir, "predictions_route.parquet")
    save_predictions_parquet(preds_df_r, preds_local_r)
    write_parquet(
        client=lakefs_client,
        repo_name=config.repo_name,
        branch_name=BRANCH_ROUTE,
        path="gold/predictions_route.parquet",
        df=preds_df_r,
    )

ref_route = lakefs_commit(
    client=lakefs_client,
    repo_name=config.repo_name,
    branch_name=BRANCH_ROUTE,
    message="Train XGBoost on route-based features, save metrics",
    metadata={"phase": "phase-6", "layer": "gold", "experiment": "route"},
)
print("Experiment B committed:", getattr(ref_route, "id", ref_route))
print("Metrics:", metrics_route)


## Section 2.4 - Comparison & Merge

TODO: Compare metrics/predictions from both experiment branches and determine winner.


In [ ]:
# TODO: Implement comparison table, overlay PR curve, and merge decision in Phase 7.
